# Label Transfer on Zeroshot NEMO Models

- Creator: Arpit Merchant (am84@sanger.ac.uk)
- Date of Creation: 3.12.2025
- Date of Last Modification: 3.12.2025 (by Arpit Merchant; am84@sanger.ac.uk)

In [1]:
%load_ext autoreload
%autoreload 2

In [56]:
import re
import os
import yaml
import tempfile
import shutil
import pickle
from tqdm import tqdm
from collections import Counter
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union, Literal

import numpy as np
import scanpy as sc
import pandas as pd
import anndata as ad
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import classification_report

import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from datasets import Dataset, load_from_disk

In [20]:
from app.helper import init_model, apply_peft, load_checkpoint
from nichejepa.models.modules import ClassificationModel
from nichejepa.datasets.cell_datasets import init_cell_dataset
from nichejepa.datasets.dataloaders import init_dataloader_and_sampler
from nichejepa.masks.block_masking import BlockMaskCollator
from nichejepa.utils.embedding import create_binary_selection_mask
from nichejepa.utils.embedding import compute_mean_unmasked_emb
from nichejepa.evaluation import knn_classifier

In [4]:
NUM_CORES = int(os.environ.get("LSB_DJOB_NUMPROC",1))
print(NUM_CORES)

2


In [5]:
# Set device
if not torch.cuda.is_available():
    device = torch.device('cpu')
else:
    device = torch.device('cuda:0')
    torch.cuda.set_device(device)
print(device)

cuda:0


# 1. Helper Functions

## 1.1 Inference

In [6]:
def load_target_encoder(
        args: Dict,
        device: str | torch.device = 'cuda:0'
    ) -> torch.nn.Module:
    """
    Load a finetuned model from checkpoint for inference.
    
    Parameters
    ----------
    args: Dict
        Zeroshot config dict

    Returns
    -------
    model : torch.nn.Module
        Loaded pretrained target encoder model
    """
    device = torch.device(device) if isinstance(device, str) else device
    
    # -------------------------------------------------------------------- #
    # LOAD MODEL CONFIG FROM PRETRAINED CHECKPOINT
    # -------------------------------------------------------------------- #
    print("Loading model config...")
    pretrained_checkpoint_path = Path(args['model']['pretrained_checkpoint_path'])
    model_config_file_path = pretrained_checkpoint_path / 'model_config.yaml'
    token_dictionary_file_path = pretrained_checkpoint_path / 'token_dictionary.pkl'
    model_checkpoint_path = pretrained_checkpoint_path / 'model_checkpoint.pt'
    
    # Load model config
    with open(model_config_file_path, 'r') as file:
        model_config = yaml.safe_load(file)
    
    # Load token dictionary
    with open(token_dictionary_file_path, 'rb') as file:
        token_dict = pickle.load(file)
    vocab_size = len(token_dict)
    n_special_values = sum(1 for key in token_dict if "spv" in key)
    
    # Get token sequence length
    n_special_tokens = len(model_config['meta']['special_tokens'])
    seq_len = (
        model_config['data']['seq_len_cell'] +
        model_config['data']['seq_len_neighborhood'] +
        n_special_tokens)
    
    # -------------------------------------------------------------------- #
    # INITIALIZE TARGET ENCODER
    # -------------------------------------------------------------------- #
    target_encoder, _ = init_model(
        gt_type=model_config['meta']['gt_type'],
        count_encoding=model_config['meta']['count_encoding'],
        n_value_bins=model_config['meta']['n_value_bins'],
        cell_pos_enc=model_config['meta']['cell_pos_enc'],
        device=device,
        vocab_size=vocab_size,
        seq_len=seq_len,
        n_special_tokens=n_special_tokens,
        n_segments=model_config['data']['n_segments'],
        n_special_values=n_special_values,
        enc_emb_dim=model_config['meta']['enc_emb_dim'],
        enc_depth=model_config['meta']['enc_depth'],
        pred_emb_dim=model_config['meta']['pred_emb_dim'],
        pred_depth=model_config['meta']['pred_depth'],
        num_heads=model_config['meta']['num_heads'],
        mlp_ratio=model_config['meta']['mlp_ratio'],
        use_flash_attention=model_config['meta']['use_flash_attention'],
        api_version=model_config['meta']['api_version'],
        sep_gene_tokens_neb=model_config['data']['sep_gene_tokens_neb'],
        predict_gene=model_config['meta']['predict_gene'],
        pos_learnable=model_config['meta']['pos_learnable'])
    
    # Load pretrained target encoder weights
    _, _, target_encoder, _, _, _, _ = load_checkpoint(
        device=device,
        r_path=model_checkpoint_path,
        encoder=None,
        predictor=None,
        target_encoder=target_encoder,
        opt=None,
        scaler=None,
        is_training=False)
    
    return target_encoder

In [7]:
def build_dataloader_for_inference(
    args: dict,
    dataset: Dataset,
    batch_size: int | None = None,
    num_workers: int | None = None,
    pin_memory: bool | None = None,
    distributed: bool = False,
    world_size: int = 1,
    rank: int = 0,
    drop_last: bool = False,
    persistent_workers: bool = False,
) -> torch.utils.data.DataLoader:
    """
    Build a dataloader for inference given args and a dataset.
    
    This function replicates the dataloader setup from finetune.py for use
    in inference scenarios.
    
    Parameters
    ----------
    args : dict
        The args dictionary containing:
        - args['model']['pretrained_checkpoint_path']: Path to pretrained model directory
        - args['data']['batch_size']: Batch size (if batch_size not provided)
        - args['data']['num_workers']: Number of workers (if num_workers not provided)
        - args['data']['pin_memory']: Pin memory flag (if pin_memory not provided)
    dataset : Dataset
        HuggingFace dataset (tokenized) to create dataloader from
    batch_size : int, optional
        Override batch size from args. If None, uses args['data']['batch_size']
    num_workers : int, optional
        Override num_workers from args. If None, uses args['data']['num_workers']
    pin_memory : bool, optional
        Override pin_memory from args. If None, uses args['data']['pin_memory']
    distributed : bool
        Whether to use distributed mode. Default: False
    world_size : int
        Number of replicas for distributed sampler. Default: 1
    rank : int
        Rank for distributed sampler. Default: 0
    drop_last : bool
        Whether to drop last incomplete batch. Default: False
    persistent_workers : bool
        Whether to keep workers alive between epochs. Default: False
    
    Returns
    -------
    loader : torch.utils.data.DataLoader
        The dataloader ready for inference
    """    
    # -------------------------------------------------------------------- #
    # LOAD MODEL CONFIG
    # -------------------------------------------------------------------- #
    print("Loading model config...")
    pretrained_checkpoint_path = Path(args['model']['pretrained_checkpoint_path'])
    model_config_file_path = pretrained_checkpoint_path / 'model_config.yaml'
    token_dictionary_file_path = pretrained_checkpoint_path / 'token_dictionary.pkl'
    
    # Load model config
    with open(model_config_file_path, 'r') as file:
        model_config = yaml.safe_load(file)
    
    # -------------------------------------------------------------------- #
    # LOAD TOKEN DICTIONARY
    # -------------------------------------------------------------------- #
    print("Loading token dictionary...")
    with open(token_dictionary_file_path, 'rb') as file:
        token_dict = pickle.load(file)
    vocab_size = len(token_dict)
    
    # Get token sequence length and number of special tokens
    n_special_tokens = len(model_config['meta']['special_tokens'])
    
    # -------------------------------------------------------------------- #
    # CREATE MASK COLLATOR
    # -------------------------------------------------------------------- #
    print("Creating mask collator...")
    mask_collator = BlockMaskCollator(
        n_targets=model_config['mask']['n_targets'],
        n_contexts=model_config['mask']['n_contexts'],
        n_segments=model_config['data']['n_segments'],
        seq_len_cell=model_config['data']['seq_len_cell'],
        seq_len_neighborhood=model_config['data']['seq_len_neighborhood'],
        n_special_tokens=n_special_tokens,
        per_block_mask_ratio=model_config['mask']['per_block_mask_ratio'],
        sample_segments=False,
        sample_gene_masks=False)
    
    # -------------------------------------------------------------------- #
    # CREATE CELL DATASET
    # -------------------------------------------------------------------- #
    print("Creating cell dataset...")
    cell_dataset = init_cell_dataset(
        dataset=dataset,
        vocab_size=vocab_size,
        seq_len_cell=model_config['data']['seq_len_cell'],
        seq_len_neighborhood=model_config['data']['seq_len_neighborhood'],
        tokenizer_type=model_config['data']['tokenizer_type'],
        gt_type=model_config['meta']['gt_type'],
        cell_pos_enc=model_config['meta']['cell_pos_enc'],
        special_tokens=model_config['meta']['special_tokens'],
        sampling_strategy=None,
        n_nonzero_tokens_list=None,
        include_cell_id=True,
        sep_gene_tokens_neb=model_config['data']['sep_gene_tokens_neb'])
    
    # -------------------------------------------------------------------- #
    # GET DATALOADER PARAMETERS
    # -------------------------------------------------------------------- #
    # Use provided values or fall back to args
    batch_size = batch_size if batch_size is not None else args['data']['batch_size']
    num_workers = num_workers if num_workers is not None else args['data']['num_workers']
    pin_memory = pin_memory if pin_memory is not None else args['data']['pin_memory']
    
    # -------------------------------------------------------------------- #
    # CREATE DATALOADER
    # -------------------------------------------------------------------- #
    print("Creating dataloader...")
    loader, _ = init_dataloader_and_sampler(
        cell_dataset=cell_dataset,
        batch_size=batch_size,
        distributed=distributed,
        world_size=world_size,
        rank=rank,
        collate_fn=mask_collator,
        pin_memory=pin_memory,
        num_workers=num_workers,
        drop_last=drop_last,
        persistent_workers=persistent_workers)
    
    return loader

In [25]:
@torch.no_grad()
def infer_with_zeroshot_model(
    model: torch.nn.Module,
    loader: torch.utils.data.DataLoader,
    args: dict,
    device: str | torch.device = 'cuda:0',
    desc: str = "Inferencing with model..."
) -> pd.DataFrame:
    """
    Run inference with the pretrained target encoder model and return embeddings as numpy array.
    
    Parameters
    ----------
    model : torch.nn.Module
        The loaded finetuned ClassificationModel (in eval mode)
    loader : torch.utils.data.DataLoader
        Dataloader created with build_dataloader_for_inference
    args : dict
        The args dictionary containing:
        - args['model']['pretrained_checkpoint_path']: Path to pretrained model directory
        - args['finetune']['selection_type']: Selection type for cell_mask
        - args['finetune']['excluded_tokens']: Excluded tokens for cell_mask
        - args['finetune']['top_k']: Top k for cell_mask
    device : str | torch.device
        Device to run inference on. Default: 'cuda:0'
    desc : str
        Description for progress bar. Default: "Inferencing with model..."
    
    Returns
    -------
    np.ndarray
        - Output of the pretrained model forward
    """
    device = torch.device(device) if isinstance(device, str) else device
    model.eval()
    
    # -------------------------------------------------------------------- #
    # LOAD MODEL CONFIG FOR INFERENCE PARAMETERS
    # -------------------------------------------------------------------- #
    pretrained_checkpoint_path = Path(args['model']['pretrained_checkpoint_path'])
    model_config_file_path = pretrained_checkpoint_path / 'model_config.yaml'
    
    with open(model_config_file_path, 'r') as file:
        model_config = yaml.safe_load(file)
    
    # Get parameters for cell_mask creation
    n_special_tokens = len(model_config['meta']['special_tokens'])
    selection_type = args['finetune']['selection_type']
    agg_excluded_tokens = args['finetune']['excluded_tokens']
    top_k = args['finetune']['top_k']
    
    # -------------------------------------------------------------------- #
    # RUN INFERENCE
    # -------------------------------------------------------------------- #
    all_cell_ids = []
    all_embeddings = []
    
    for itr, (udata, _, _, masks_attention) in tqdm(enumerate(loader), desc=desc):
        for key in udata.keys():
            if key != 'cell_id':
                udata[key] = udata[key].to(device, non_blocking=True)
        masks_attention = masks_attention.to(device, non_blocking=True)
        
        # Get non-special tokens for cell_mask creation
        ns_tokens = udata['tokens'][:, n_special_tokens:]
        
        # Create cell mask for aggregation
        cell_mask = create_binary_selection_mask(
            ns_tokens,
            selection_type=selection_type,
            excluded_tokens=agg_excluded_tokens,
            seq_len_cell=model_config['data']['seq_len_cell'],
            top_k=top_k,
        )
        
        # Forward pass
        # h.shape = (<batch_size>, <seq_len>, <embedding_dim>)
        h, _= model(
                    batch=udata,
                    masks_attention=masks_attention
                )
        
        # Aggregate gene embeddings into cell embeddings
        # h.shape = (<batch_size>, <embedding_dim>)
        h = compute_mean_unmasked_emb(
                h,
                cell_mask
            )
        
        # Collect cell IDs and embeddings
        all_cell_ids.extend(udata['cell_id'])
        all_embeddings.append(h.cpu())
    
    # Concatenate all embeddings
    all_embeddings = torch.cat(all_embeddings, dim=0).numpy()
    
    return all_cell_ids, all_embeddings

In [38]:
def collect_labels(
    args: Dict,
    adata: ad.AnnData,
    cell_ids: list[str]
):
    
    label_name = args['data']['label_name']
    
    # Assert required columns exist in train_adata.obs
    assert 'cell_id' in adata.obs.columns, f"'cell_id' not found in adata.obs.columns. Available columns: {adata.obs.columns.tolist()}"
    assert label_name in adata.obs.columns, f"'{label_name}' not found in adata.obs.columns. Available columns: {adata.obs.columns.tolist()}"
    
    # Assert that label_name contains integer values (required for CrossEntropyLoss)
    label_dtype = adata.obs[label_name].dtype
    assert pd.api.types.is_integer_dtype(label_dtype), (
        f"Labels in '{label_name}' must be integers, but got dtype: {label_dtype}. "
        f"Please encode string labels to integer class indices."
    )
    
    # Create label lookup dictionary indexed by cell_id (do once before training loop)
    label_lookup = adata.obs.set_index('cell_id')[label_name]
    labels = np.array(label_lookup[cell_ids].values, dtype=np.int64)
    
    return labels

# 2. Task I: Niche Transfer

## 2.1 3K Skin

### 2.1.1 Load Data

In [9]:
train_adata = sc.read_h5ad("/lustre/scratch126/cellgen/lotfollahi/am84/NEMO/data/xhs_niche-classification-3K/train.h5ad")
val_adata = sc.read_h5ad("/lustre/scratch126/cellgen/lotfollahi/am84/NEMO/data/xhs_niche-classification-3K/val.h5ad")
test_adata = sc.read_h5ad("/lustre/scratch126/cellgen/lotfollahi/am84/NEMO/data/xhs_niche-classification-3K/test.h5ad")

In [15]:
train_ds = load_from_disk("/lustre/scratch126/cellgen/lotfollahi/am84/NEMO/data/xhs_niche-classification-3K/train.dataset")
train_ds.set_format(
    type="torch",
    columns=[c for c in train_ds.column_names],
    output_all_columns=True
)
    
val_ds = load_from_disk("/lustre/scratch126/cellgen/lotfollahi/am84/NEMO/data/xhs_niche-classification-3K/val.dataset")
val_ds.set_format(
    type="torch",
    columns=[c for c in val_ds.column_names],
    output_all_columns=True
)

test_ds = load_from_disk("/lustre/scratch126/cellgen/lotfollahi/am84/NEMO/data/xhs_niche-classification-3K/test.dataset")
test_ds.set_format(
    type="torch",
    columns=[c for c in test_ds.column_names],
    output_all_columns=True
)

### 2.1.2 Load Pretrained Target Encoder

In [11]:
zeroshot_config_file = "/lustre/scratch126/cellgen/lotfollahi/am84/NEMO/reproducibility/config/finetuning/xhs_niche-classification-3K_zero-shot.yaml"
with open(zeroshot_config_file, 'r') as fp:
    zsl_args = yaml.safe_load(fp)

In [12]:
target_encoder = load_target_encoder(
                    zsl_args
                )

Loading model config...
INFO:root:EncoderMultiMaskWrapper(
  (backbone): GeneTransformerCombinedEncoder(
    (token_embed): Embedding(23407, 384, padding_idx=0)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
    (pos_embed): E

### 2.1.3 Build Train-Val-Test Dataloaders

In [17]:
# Build dataloader
train_loader = build_dataloader_for_inference(
    args=zsl_args,
    dataset=train_ds,
    batch_size=100,  # Optional: override batch size
    num_workers=NUM_CORES,  # Optional: override num_workers
)

# Build dataloader
val_loader = build_dataloader_for_inference(
    args=zsl_args,
    dataset=val_ds,
    batch_size=100,  # Optional: override batch size
    num_workers=NUM_CORES,  # Optional: override num_workers
)

# Build dataloader
test_loader = build_dataloader_for_inference(
    args=zsl_args,
    dataset=test_ds,
    batch_size=100,  # Optional: override batch size
    num_workers=NUM_CORES,  # Optional: override num_workers
)

Loading model config...
Loading token dictionary...
Creating mask collator...
Creating cell dataset...
Creating dataloader...
INFO:root:Dataloader and -sampler created.
Loading model config...
Loading token dictionary...
Creating mask collator...
Creating cell dataset...
Creating dataloader...
INFO:root:Dataloader and -sampler created.
Loading model config...
Loading token dictionary...
Creating mask collator...
Creating cell dataset...
Creating dataloader...
INFO:root:Dataloader and -sampler created.


### 2.1.4 Target Encoder Inference

In [27]:
train_cell_ids, train_features = infer_with_zeroshot_model(
                                        model=target_encoder,
                                        loader=train_loader,
                                        args=zsl_args,
                                        device=device,
                                        desc="Train Features..."
                                    )

Train Features...: 30it [05:20, 10.67s/it]


In [30]:
with open("/lustre/scratch126/cellgen/lotfollahi/am84/NEMO/data/xhs_niche-classification-3K/train_features.npy", 'wb') as f:
    np.save(f, train_features)

In [31]:
test_cell_ids, test_features = infer_with_zeroshot_model(
                                        model=target_encoder,
                                        loader=test_loader,
                                        args=zsl_args,
                                        device=device,
                                        desc="Test Features..."
                                    )

Test Features...: 30it [05:22, 10.76s/it]


In [32]:
with open("/lustre/scratch126/cellgen/lotfollahi/am84/NEMO/data/xhs_niche-classification-3K/test_features.npy", 'wb') as f:
    np.save(f, test_features)

In [ ]:
with open("/lustre/scratch126/cellgen/lotfollahi/am84/NEMO/data/xhs_niche-classification-3K/train_features.npy", 'rb') as f:
    train_features = np.load(f)

In [ ]:
with open("/lustre/scratch126/cellgen/lotfollahi/am84/NEMO/data/xhs_niche-classification-3K/test_features.npy", 'rb') as f:
    test_features = np.load(f)

In [39]:
train_labels = collect_labels(
                    args=zsl_args,
                    adata=train_adata,
                    cell_ids=train_cell_ids
                )

In [40]:
train_labels

array([ 3,  5,  5, ..., 25, 25, 25], shape=(3000,))

In [41]:
test_labels = collect_labels(
                    args=zsl_args,
                    adata=test_adata,
                    cell_ids=test_cell_ids
                )

In [66]:
k = 4
predictions = knn_classifier(
        features_train=train_features,
        labels_train=train_labels,
        features_test=test_features,
        labels_test=test_labels,
        k=k
    )

Using device: cuda

--- Evaluation Report on Test Set ---
              precision    recall  f1-score   support

           3     0.6747    0.9440    0.7869       268
           5     0.9554    0.9427    0.9490       227
           6     0.5294    0.6923    0.6000        13
          10     0.3810    0.6400    0.4776        25
          11     0.7143    0.1667    0.2703        30
          12     0.9598    0.8566    0.9053       279
          14     0.6058    0.6068    0.6063       651
          15     0.7600    0.5278    0.6230        36
          17     0.6118    0.8611    0.7154       216
          18     0.6452    0.3509    0.4545       114
          20     0.9600    0.9600    0.9600        25
          22     0.8289    0.7432    0.7837       958
          24     0.9831    0.9667    0.9748       120
          25     0.2927    0.3158    0.3038        38

    accuracy                         0.7467      3000
   macro avg     0.7073    0.6839    0.6722      3000
weighted avg     0.759

In [65]:
cls_report = classification_report(
        [test_adata.uns['niche19_idx2label'][str(i)] for i in test_labels],
        [test_adata.uns['niche19_idx2label'][str(i)] for i in predictions],
        digits=4
)
print(cls_report)

                          precision    recall  f1-score   support

         Epidermis_basal     0.6334    0.9478    0.7593       268
          Epidermis_late     0.9478    0.9604    0.9540       227
           Epidermis_mid     0.6000    0.6923    0.6429        13
              Hypodermis     0.0000    0.0000    0.0000        25
                Large_BV     0.0000    0.0000    0.0000        30
                  Muscle     0.9514    0.8423    0.8935       279
        Papillary_dermis     0.5701    0.5806    0.5753       651
              Perineural     0.0000    0.0000    0.0000        36
        Reticular_dermis     0.6159    0.8981    0.7307       216
Reticular_dermis_LE_rich     0.6667    0.1754    0.2778       114
         Sebaceous_gland     0.9600    0.9600    0.9600        25
                Small_BV     0.7957    0.7683    0.7817       958
     Sweat_gland_channel     0.9829    0.9583    0.9705       120
              Tzone-like     0.4231    0.2895    0.3438        38

        

/software/cellgen/team361/am84/envs/nemo/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/software/cellgen/team361/am84/envs/nemo/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/software/cellgen/team361/am84/envs/nemo/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mod

In [58]:
test_adata.uns['niche19_idx2label']

{'0': 'EpidermisInflamm_late',
 '1': 'EpidermisInflamm_mid',
 '10': 'Hypodermis',
 '11': 'Large_BV',
 '12': 'Muscle',
 '13': 'Nonspecific/folded',
 '14': 'Papillary_dermis',
 '15': 'Perineural',
 '16': 'Plasma_cell_rich',
 '17': 'Reticular_dermis',
 '18': 'Reticular_dermis_LE_rich',
 '19': 'Sebaceous_duct',
 '2': 'Epidermis_APChi',
 '20': 'Sebaceous_gland',
 '21': 'Sebaceous_immune',
 '22': 'Small_BV',
 '23': 'Sweat_gland',
 '24': 'Sweat_gland_channel',
 '25': 'Tzone-like',
 '26': 'VenuleMuscle',
 '3': 'Epidermis_basal',
 '4': 'Epidermis_basal_cycling',
 '5': 'Epidermis_late',
 '6': 'Epidermis_mid',
 '7': 'HF_inner',
 '8': 'HF_innermost',
 '9': 'HF_outer'}